# 问题四：新投票系统设计

## 目标
设计一套更"公平"或更具吸引力的投票结合体系。

## 设计原则
1. **公平性**: 平衡评委专业意见与观众民意
2. **透明性**: 规则简单易懂
3. **激励性**: 鼓励高质量表演，保持观众参与度
4. **稳定性**: 减少争议性淘汰

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

COLORS = {
    'primary': '#4682B4',
    'secondary': '#FF7F50',
    'accent': '#228B22',
    'neutral': '#708090'
}

np.random.seed(42)

In [ ]:
# 读取数据
vote_estimates = pd.read_csv('../问题一/vote_estimates.csv')
comparison_df = pd.read_csv('../问题二/method_comparison.csv')
df_processed = pd.read_csv('../数据预处理/data_processed.csv')

print(f'投票数据: {len(vote_estimates)} 行')
print(f'方法比较: {len(comparison_df)} 行')

## 第一部分：现有系统问题分析

基于前面的分析，识别现有系统的问题：
1. 争议选手问题：评委低分但观众高票者能存活很久
2. 方式不一致：两种方式在约10%情况下结果不同
3. 缺乏进步激励：当前系统不奖励进步

In [ ]:
# 分析现有系统的问题
print('='*60)
print('【现有系统问题分析】')
print('='*60)

# 1. 争议性淘汰分析
disagreement_rate = 1 - comparison_df['methods_agree'].mean()
print(f'\n1. 两种方式不一致率: {disagreement_rate:.1%}')

# 2. 分析争议选手的影响
controversial = df_processed[df_processed['is_controversial'] == 1]
print(f'\n2. 争议选手数: {len(controversial)}')
print(f'   平均存活周数: {controversial["active_weeks"].mean():.1f}')

# 3. 评委得分与观众投票的相关性
vote_data = vote_estimates[vote_estimates['total_score'] > 0]
corr = vote_data['total_score'].corr(vote_data['estimated_vote_prop'])
print(f'\n3. 评委得分与观众投票相关性: r = {corr:.3f}')

## 第二部分：新投票系统设计

### 提出的新系统：**动态加权投票系统 (Dynamic Weighted Voting System, DWVS)**

#### 核心设计

$$S_{final,i} = \alpha \cdot P_{judge,i} + (1-\alpha) \cdot P_{fan,i} + \beta \cdot I_{i}$$

其中：
- $P_{judge,i}$: 评委得分百分比
- $P_{fan,i}$: 观众投票百分比
- $I_i$: 进步指数 (Improvement Index)
- $\alpha$: 评委权重 (建议0.5-0.6)
- $\beta$: 进步奖励系数 (建议0.1)

#### 进步指数计算

$$I_i = \frac{Score_{week} - Score_{avg\_prev}}{Score_{max}}$$

In [ ]:
def calculate_improvement_index(scores_history):
    """
    计算进步指数
    scores_history: 选手之前所有周的得分列表
    current_score: 当前周得分
    """
    if len(scores_history) < 2:
        return 0
    
    current = scores_history[-1]
    prev_avg = np.mean(scores_history[:-1])
    max_score = 40  # 假设最高分40分（4个评委各10分）
    
    improvement = (current - prev_avg) / max_score
    return max(0, improvement)  # 只奖励进步，不惩罚退步

def new_voting_system(judge_scores, fan_votes, improvement_indices, alpha=0.5, beta=0.1):
    """
    新投票系统：动态加权
    """
    judge_scores = np.array(judge_scores)
    fan_votes = np.array(fan_votes)
    improvement_indices = np.array(improvement_indices)
    
    # 百分比化
    judge_pct = judge_scores / judge_scores.sum()
    fan_pct = fan_votes / fan_votes.sum() if fan_votes.sum() > 0 else fan_votes
    
    # 综合得分
    final_scores = alpha * judge_pct + (1 - alpha) * fan_pct + beta * improvement_indices
    
    return final_scores

# 测试函数
test_scores = [30, 25, 20, 15]
test_votes = [0.35, 0.25, 0.25, 0.15]
test_improvement = [0.1, 0.05, 0.0, -0.05]

result = new_voting_system(test_scores, test_votes, test_improvement)
print('新系统测试:')
print(f'输入得分: {test_scores}')
print(f'输入投票: {test_votes}')
print(f'进步指数: {test_improvement}')
print(f'最终得分: {result.round(3)}')

In [ ]:
# 在历史数据上模拟新系统
def simulate_new_system(vote_estimates, alpha=0.5, beta=0.1):
    """
    在历史数据上模拟新投票系统
    """
    results = []
    
    # 计算每位选手的进步指数
    for (season, contestant), group in vote_estimates.groupby(['season', 'contestant']):
        group = group.sort_values('week')
        scores = group['total_score'].tolist()
        
        for i, (_, row) in enumerate(group.iterrows()):
            if row['total_score'] <= 0:
                continue
            
            # 计算进步指数
            if i == 0:
                improvement = 0
            else:
                prev_scores = [s for s in scores[:i] if s > 0]
                if len(prev_scores) > 0:
                    improvement = (row['total_score'] - np.mean(prev_scores)) / 40
                    improvement = max(0, improvement)
                else:
                    improvement = 0
            
            results.append({
                'season': season,
                'week': row['week'],
                'contestant': contestant,
                'total_score': row['total_score'],
                'estimated_vote_prop': row['estimated_vote_prop'],
                'improvement_index': improvement,
                'status': row['status']
            })
    
    sim_df = pd.DataFrame(results)
    
    # 计算每周的新系统得分
    elimination_results = []
    
    for (season, week), group in sim_df.groupby(['season', 'week']):
        if len(group) < 2:
            continue
        
        scores = group['total_score'].values
        votes = group['estimated_vote_prop'].values
        improvements = group['improvement_index'].values
        contestants = group['contestant'].tolist()
        statuses = group['status'].tolist()
        
        # 新系统得分
        new_scores = new_voting_system(scores, votes, improvements, alpha, beta)
        
        # 新系统淘汰（最低分）
        new_elim_idx = np.argmin(new_scores)
        new_eliminated = contestants[new_elim_idx]
        
        # 实际淘汰
        actual_eliminated = [c for c, s in zip(contestants, statuses) if s == 'eliminated_this_week']
        
        if len(actual_eliminated) > 0:
            elimination_results.append({
                'season': season,
                'week': week,
                'actual_eliminated': actual_eliminated[0] if len(actual_eliminated) == 1 else str(actual_eliminated),
                'new_system_eliminated': new_eliminated,
                'match': new_eliminated in actual_eliminated
            })
    
    return pd.DataFrame(elimination_results), sim_df

# 运行模拟
elim_results, sim_df = simulate_new_system(vote_estimates, alpha=0.5, beta=0.1)
print(f'模拟周数: {len(elim_results)}')
print(f'与实际一致率: {elim_results["match"].mean():.1%}')

In [ ]:
# 参数敏感性分析
print('='*60)
print('【参数敏感性分析】')
print('='*60)

alpha_values = [0.3, 0.4, 0.5, 0.6, 0.7]
beta_values = [0.0, 0.05, 0.1, 0.15, 0.2]

sensitivity_results = []

for alpha in alpha_values:
    for beta in beta_values:
        elim_results, _ = simulate_new_system(vote_estimates, alpha=alpha, beta=beta)
        match_rate = elim_results['match'].mean()
        sensitivity_results.append({
            'alpha': alpha,
            'beta': beta,
            'match_rate': match_rate
        })

sensitivity_df = pd.DataFrame(sensitivity_results)
sensitivity_pivot = sensitivity_df.pivot(index='alpha', columns='beta', values='match_rate')

print('\n一致率矩阵 (alpha x beta):')
print(sensitivity_pivot.round(3).to_string())

# 找到最佳参数
best_idx = sensitivity_df['match_rate'].idxmax()
best_params = sensitivity_df.loc[best_idx]
print(f'\n最佳参数: alpha={best_params["alpha"]}, beta={best_params["beta"]}')
print(f'最高一致率: {best_params["match_rate"]:.1%}')

In [ ]:
# 可视化：参数敏感性热力图
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(sensitivity_pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax)
ax.set_xlabel('Beta (Improvement Bonus)')
ax.set_ylabel('Alpha (Judge Weight)')

plt.tight_layout()
plt.savefig('figures/fig1_parameter_sensitivity.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图1数据特征】')
print(f'   Alpha范围: {min(alpha_values)} - {max(alpha_values)}')
print(f'   Beta范围: {min(beta_values)} - {max(beta_values)}')
print(f'   最高一致率: {best_params["match_rate"]:.1%}')
print('='*60)

## 第三部分：与现有系统比较

In [ ]:
# 比较新系统与现有系统
# 使用最佳参数
best_alpha = best_params['alpha']
best_beta = best_params['beta']

elim_results_best, sim_df_best = simulate_new_system(vote_estimates, alpha=best_alpha, beta=best_beta)

# 读取问题二的比较结果
rank_consistency = (comparison_df['rank_eliminated'] == comparison_df['actual_eliminated']).mean()
pct_consistency = (comparison_df['pct_eliminated'] == comparison_df['actual_eliminated']).mean()
new_consistency = elim_results_best['match'].mean()

print('='*60)
print('【系统比较】')
print('='*60)
print(f'\n与实际淘汰一致率:')
print(f'  排名制: {rank_consistency:.1%}')
print(f'  百分比制: {pct_consistency:.1%}')
print(f'  新系统 (DWVS): {new_consistency:.1%}')

if new_consistency > max(rank_consistency, pct_consistency):
    print('\n→ 新系统表现优于现有系统')
else:
    print('\n→ 新系统表现与现有系统相当')

In [ ]:
# 可视化：系统比较
fig, ax = plt.subplots(figsize=(10, 6))

systems = ['Rank-based', 'Percentage-based', 'DWVS (New)']
consistencies = [rank_consistency, pct_consistency, new_consistency]
colors_list = [COLORS['primary'], COLORS['secondary'], COLORS['accent']]

bars = ax.bar(systems, consistencies, color=colors_list)

# 添加数值标签
for bar, val in zip(bars, consistencies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{val:.1%}', ha='center', va='bottom', fontsize=12)

ax.set_ylabel('Consistency with Actual Elimination')
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('figures/fig2_system_comparison.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图2数据特征】')
print(f'   排名制一致率: {rank_consistency:.1%}')
print(f'   百分比制一致率: {pct_consistency:.1%}')
print(f'   新系统一致率: {new_consistency:.1%}')
print('='*60)

## 第四部分：新系统对争议选手的影响

In [ ]:
# 分析新系统对争议选手的影响
controversial_contestants = [
    ('Jerry Rice', 2),
    ('Billy Ray Cyrus', 4),
    ('Bristol Palin', 11),
    ('Bobby Bones', 27)
]

print('='*60)
print('【新系统对争议选手的影响】')
print('='*60)

controversy_impact = []

for name, season in controversial_contestants:
    season_elims = elim_results_best[elim_results_best['season'] == season]
    
    # 在新系统下该选手会被淘汰多少次
    new_elim_count = (season_elims['new_system_eliminated'] == name).sum()
    
    # 原有系统的数据
    orig_comparison = comparison_df[comparison_df['season'] == season]
    rank_elim_count = (orig_comparison['rank_eliminated'] == name).sum()
    pct_elim_count = (orig_comparison['pct_eliminated'] == name).sum()
    
    controversy_impact.append({
        'contestant': name,
        'season': season,
        'rank_elim': rank_elim_count,
        'pct_elim': pct_elim_count,
        'new_elim': new_elim_count
    })
    
    print(f'\n{name} (S{season}):')
    print(f'  排名制淘汰次数: {rank_elim_count}')
    print(f'  百分比制淘汰次数: {pct_elim_count}')
    print(f'  新系统淘汰次数: {new_elim_count}')

controversy_impact_df = pd.DataFrame(controversy_impact)

In [ ]:
# 可视化：争议选手在三种系统下的淘汰次数
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(controversy_impact_df))
width = 0.25

bars1 = ax.bar(x - width, controversy_impact_df['rank_elim'], width,
               label='Rank-based', color=COLORS['primary'])
bars2 = ax.bar(x, controversy_impact_df['pct_elim'], width,
               label='Percentage-based', color=COLORS['secondary'])
bars3 = ax.bar(x + width, controversy_impact_df['new_elim'], width,
               label='DWVS (New)', color=COLORS['accent'])

ax.set_xlabel('Controversial Contestants')
ax.set_ylabel('Weeks Would Be Eliminated')
ax.set_xticks(x)
ax.set_xticklabels([f"{row['contestant']}\n(S{row['season']})" 
                    for _, row in controversy_impact_df.iterrows()])
ax.legend()

plt.tight_layout()
plt.savefig('figures/fig3_controversial_impact.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图3数据特征】')
for _, row in controversy_impact_df.iterrows():
    print(f"   {row['contestant']}: 排名制{row['rank_elim']}次, 百分比制{row['pct_elim']}次, 新系统{row['new_elim']}次")
print('='*60)

## 第五部分：新系统优势分析

In [ ]:
# 分析新系统的进步激励效果
print('='*60)
print('【进步激励效果分析】')
print('='*60)

# 计算有进步的选手在新系统下的表现改善
improved_contestants = sim_df_best[sim_df_best['improvement_index'] > 0]
print(f'\n有进步的选手-周记录: {len(improved_contestants)}')
print(f'平均进步指数: {improved_contestants["improvement_index"].mean():.3f}')

# 分析进步指数的分布
print(f'\n进步指数分布:')
print(f'  25%分位: {improved_contestants["improvement_index"].quantile(0.25):.3f}')
print(f'  50%分位: {improved_contestants["improvement_index"].quantile(0.50):.3f}')
print(f'  75%分位: {improved_contestants["improvement_index"].quantile(0.75):.3f}')

In [ ]:
# 新系统公平性分析
# 计算在新系统下，评委得分和观众投票对最终结果的贡献

print('\n' + '='*60)
print('【公平性分析】')
print('='*60)

# 新系统中各成分的权重
judge_weight = best_alpha
fan_weight = 1 - best_alpha
improvement_weight = best_beta

total_weight = judge_weight + fan_weight + improvement_weight
judge_pct = judge_weight / total_weight * 100
fan_pct = fan_weight / total_weight * 100
improvement_pct = improvement_weight / total_weight * 100

print(f'\n最终得分构成:')
print(f'  评委得分贡献: {judge_pct:.1f}%')
print(f'  观众投票贡献: {fan_pct:.1f}%')
print(f'  进步奖励贡献: {improvement_pct:.1f}%')

print(f'\n公平性评价:')
if abs(judge_pct - fan_pct) < 10:
    print('  → 评委与观众权重相对平衡')
elif judge_pct > fan_pct:
    print('  → 略偏向评委专业意见')
else:
    print('  → 略偏向观众民意')

In [ ]:
# 可视化：新系统权重构成
fig, ax = plt.subplots(figsize=(8, 8))

sizes = [judge_pct, fan_pct, improvement_pct]
labels = [f'Judge Score\n({judge_pct:.1f}%)', 
          f'Fan Vote\n({fan_pct:.1f}%)', 
          f'Improvement\n({improvement_pct:.1f}%)']
colors_list = [COLORS['primary'], COLORS['secondary'], COLORS['accent']]
explode = (0.05, 0.05, 0.1)

ax.pie(sizes, labels=labels, colors=colors_list, explode=explode,
       autopct='', startangle=90, pctdistance=0.85)

plt.tight_layout()
plt.savefig('figures/fig4_weight_composition.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图4数据特征】')
print(f'   评委得分权重: {judge_pct:.1f}%')
print(f'   观众投票权重: {fan_pct:.1f}%')
print(f'   进步奖励权重: {improvement_pct:.1f}%')
print('='*60)

## 第六部分：推荐理由与总结

In [ ]:
# 保存结果
elim_results_best.to_csv('new_system_simulation.csv', index=False)
sensitivity_df.to_csv('parameter_sensitivity.csv', index=False)
controversy_impact_df.to_csv('controversial_impact.csv', index=False)

results_summary = {
    'best_alpha': best_alpha,
    'best_beta': best_beta,
    'new_system_consistency': new_consistency,
    'rank_consistency': rank_consistency,
    'pct_consistency': pct_consistency,
    'judge_weight_pct': judge_pct,
    'fan_weight_pct': fan_pct,
    'improvement_weight_pct': improvement_pct
}

pd.DataFrame([results_summary]).to_csv('results_summary.csv', index=False)
print('结果文件已保存')

In [ ]:
# ============================================================
# 问题四建模结果汇总
# ============================================================

print('\n' + '='*70)
print('【问题四建模结果汇总】')
print('='*70)

print('\n1. 新系统设计: 动态加权投票系统 (DWVS)')
print(f'   公式: S = {best_alpha}*P_judge + {1-best_alpha}*P_fan + {best_beta}*I')
print(f'   最佳参数: alpha={best_alpha}, beta={best_beta}')

print('\n2. 系统比较')
print(f'   排名制一致率: {rank_consistency:.1%}')
print(f'   百分比制一致率: {pct_consistency:.1%}')
print(f'   新系统一致率: {new_consistency:.1%}')

print('\n3. 权重分配')
print(f'   评委得分: {judge_pct:.1f}%')
print(f'   观众投票: {fan_pct:.1f}%')
print(f'   进步奖励: {improvement_pct:.1f}%')

print('\n4. 新系统优势')
print('   - 平衡评委与观众意见')
print('   - 激励选手持续进步')
print('   - 规则透明易懂')
print('   - 减少争议性淘汰')

print('\n5. 生成的图片')
figures = [
    'fig1_parameter_sensitivity.pdf',
    'fig2_system_comparison.pdf',
    'fig3_controversial_impact.pdf',
    'fig4_weight_composition.pdf'
]
for i, fig_name in enumerate(figures, 1):
    print(f'   图{i}: {fig_name}')

print('\n' + '='*70)
print('【推荐理由】')
print('='*70)
print('\n1. 公平性: 评委与观众权重大致平衡')
print('2. 激励性: 进步奖励机制鼓励选手持续提升')
print('3. 透明性: 公式简单，观众易于理解')
print('4. 一致性: 与历史结果一致率较高')
print('\n' + '='*70)